# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


In [ ]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [ ]:
# Task 1 — Heatmap: content by rating and release decade
# ── Data prep ─────────────────────────────────────────────────────────────────
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

keep_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']     # most common ratings
heat = df[df['rating'].isin(keep_ratings)].copy()

# Count titles per rating × decade, then pivot into a matrix
matrix = (
    heat.groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
    .pivot(index='rating', columns='decade', values='count')
    .reindex(keep_ratings)              # keep ratings in a sensible fixed order
    .sort_index(axis=1)                 # decades left→right in time order
)

# ── Heatmap (counts → SEQUENTIAL scale: Blues) ────────────────────────────────
fig = px.imshow(
    matrix,
    text_auto=True,                     # show the count inside each cell
    color_continuous_scale='Blues',     # sequential: data is non-negative counts
    aspect='auto',
    labels={'x': 'Release decade', 'y': 'Content rating', 'color': 'Titles'},
    title='TV-MA defines the streaming era \u2014 mature content explodes in the 2010s while older decades skew PG-13/R',
)

# Annotate the single hottest cell directly
peak_val = matrix.max().max()
peak_rating = matrix.max(axis=1).idxmax()
peak_decade = matrix.loc[peak_rating].idxmax()
fig.add_annotation(
    x=peak_decade, y=peak_rating,
    text=f'Peak: {int(peak_val)} {peak_rating}<br>titles in the {peak_decade}',
    showarrow=True, arrowhead=2, arrowcolor='#D55E00', arrowwidth=2,
    ax=70, ay=-55,
    font=dict(color='#D55E00', size=12, family='Arial'),
    bgcolor='white', bordercolor='#D55E00', borderwidth=1, borderpad=4,
)

fig.update_traces(textfont=dict(size=12))
fig.update_layout(
    font=dict(family='Arial', size=12),
    title=dict(font=dict(size=15), x=0.01, xanchor='left'),
    paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(l=10, r=10, t=60, b=10),
    width=900, height=520,
)
fig.update_xaxes(side='bottom')
fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [ ]:
# Task 2 — Waterfall: Movie library growth by year
# ── Data prep ─────────────────────────────────────────────────────────────────
movies = df[df['type'] == 'Movie'].copy()

per_year = (
    movies[movies['added_year'].between(2015, 2022)]
    .groupby('added_year')
    .size()
    .reindex(range(2015, 2023), fill_value=0)
    .reset_index(name='added')
    .rename(columns={'index': 'added_year'})
)

years   = [str(y) for y in per_year['added_year']]
adds    = per_year['added'].tolist()
total   = int(per_year['added'].sum())

# Build the waterfall: yearly additions (relative) + a final cumulative Total bar
x_labels = years + ['Total']
measures = ['relative'] * len(years) + ['total']
y_values = adds + [total]

# ── Waterfall (green = additions, blue = total) ───────────────────────────────
fig = go.Figure(go.Waterfall(
    orientation='v',
    measure=measures,
    x=x_labels,
    y=y_values,
    text=[f'+{v:,}' for v in adds] + [f'{total:,}'],
    textposition='outside',
    connector={'line': {'color': '#BBBBBB'}},
    increasing={'marker': {'color': '#2E8B57'}},   # green — additions
    decreasing={'marker': {'color': '#C0392B'}},   # red   — (none here, but per the rules)
    totals={'marker': {'color': '#0072B2'}},       # blue  — cumulative total
))

# Annotate the year with the largest single addition
peak_i   = int(per_year['added'].idxmax())
peak_yr  = years[peak_i]
peak_add = adds[peak_i]
fig.add_annotation(
    x=peak_yr, y=peak_add,
    text=f'Biggest single year:<br>+{peak_add:,} movies in {peak_yr}',
    showarrow=True, arrowhead=2, arrowcolor='#2E8B57', arrowwidth=2,
    ax=0, ay=-70,
    font=dict(color='#2E8B57', size=12, family='Arial'),
    bgcolor='white', bordercolor='#2E8B57', borderwidth=1, borderpad=4,
)

fig.update_layout(
    title=dict(
        text=f'Netflix front-loaded its movie catalogue \u2014 {peak_yr} added the most, '
             f'building to {total:,} titles by 2022',
        font=dict(size=15), x=0.01, xanchor='left',
    ),
    font=dict(family='Arial', size=12),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=20, r=20, t=60, b=40),
    width=950, height=560,
    yaxis_title='Movies added', xaxis_title='Year added',
    showlegend=False,
)
fig.update_yaxes(gridcolor='#EEEEEE', zeroline=False)
fig.show()
